# Exp19 — Preflight gate for the remaining tasks (negation, antonym, agreement, gender, classifier)

Companion to Exp18 (IOI). For every (model, task, language) cell this reports whether the task is **measurable**: items tokenize cleanly, the candidates are scorable, and baseline accuracy clears the gate.

| Task | Predicted ε | Languages |
|---|---|---|
| Negation (neg + affirmative control, antonym-pair completion) | mid | EN FR ES DE ZH AR HI |
| Antonym (lexical semantics) | low | EN FR ES DE ZH AR HI |
| Agreement (number, congruent vs attractor-incongruent) | mid-high | EN FR ES DE AR HI (ZH has none) |
| Gender (adjective agreement; typical vs opaque nouns) | high | FR ES AR HI (EN/ZH have none; DE excluded, see Cell 1) |
| Classifier (numeral classifier choice) | high, ZH-specific | ZH |

Scoring: full-sequence log-prob of each candidate (not first token), margin = correct − foil. Gate: accuracy >= 0.75, Wilson 95% lower bound > 0.5 (n per cell is small, 6 to 48), and >= 90% of items scorable.

All items were written by hand and need a native-speaker read. Accuracy near 1.0 means the task cannot separate languages or sizes by accuracy; use the margins.

In [1]:
# CELL 1 — config + data (edit MODELS / GPU_ID only)
import os
GPU_ID = 2
os.environ.setdefault("CUDA_VISIBLE_DEVICES", str(GPU_ID))
import json, math, random
from pathlib import Path
OUT = Path("preflight_results"); OUT.mkdir(exist_ok=True)

MODELS = {"bloom-1b7": "bigscience/bloom-1b7", "bloom-7b1": "bigscience/bloom-7b1"}   # add "gemma-2-9b": "/home/models/gemma-2-9b"
def prepend_bos(key): return "gemma" in key
SEP = {"zh": ""}                       # string between context and candidate (default " ")
ACC_GATE, COVER_GATE = 0.75, 0.90

# ---- NEGATION: (s1, s2, adj_a, adj_b). Items per pair: neg(a)->b, neg(b)->a, aff(a)->a, aff(b)->b
NEG_FMT = {
 "en": ("{s1} is not {x}, so {s2} is",        "{s1} is {x}, so {s2} is"),
 "fr": ("{s1} n'est pas {x}, donc {s2} est",  "{s1} est {x}, donc {s2} est"),
 "es": ("{s1} no está {x}, así que {s2} está","{s1} está {x}, así que {s2} está"),
 "de": ("{s1} ist nicht {x}, also ist {s2}",  "{s1} ist {x}, also ist {s2}"),
 "zh": ("{s1}不是{x}，所以{s2}是",              "{s1}是{x}，所以{s2}是"),
 "hi": ("{s1} {x} नहीं है, इसलिए {s2}",        "{s1} {x} है, इसलिए {s2}"),
}
NEG = {
 "en": [("The door","the door","open","closed"),("The glass","the glass","full","empty"),("The soup","the soup","hot","cold"),("The towel","the towel","wet","dry"),("The room","the room","clean","dirty")],
 "fr": [("La porte","la porte","ouverte","fermée"),("Le verre","le verre","plein","vide"),("La soupe","la soupe","chaude","froide"),("La serviette","la serviette","mouillée","sèche"),("La chambre","la chambre","propre","sale")],
 "es": [("La puerta","la puerta","abierta","cerrada"),("El vaso","el vaso","lleno","vacío"),("La sopa","la sopa","caliente","fría"),("La toalla","la toalla","mojada","seca"),("La habitación","la habitación","limpia","sucia")],
 "de": [("Die Tür","die Tür","offen","geschlossen"),("Das Glas","das Glas","voll","leer"),("Die Suppe","die Suppe","heiß","kalt"),("Das Handtuch","das Handtuch","nass","trocken"),("Das Zimmer","das Zimmer","sauber","schmutzig")],
 "zh": [("门","门","开着的","关着的"),("杯子","杯子","满的","空的"),("汤","汤","热的","冷的"),("毛巾","毛巾","湿的","干的"),("房间","房间","干净的","脏的")],
 "hi": [("दरवाज़ा","दरवाज़ा","खुला","बंद"),("गिलास","गिलास","भरा","खाली"),("सूप","सूप","गरम","ठंडा"),("तौलिया","तौलिया","गीला","सूखा"),("कमरा","कमरा","साफ़","गंदा")],
}
# Arabic needs the accusative predicate after ليس: (NP, ليس/ليست, a_nom, b_nom, a_acc, b_acc)
NEG_AR = [("الباب","ليس","مفتوح","مغلق","مفتوحا","مغلقا"),("الكأس","ليس","ممتلئ","فارغ","ممتلئا","فارغا"),
          ("الحساء","ليس","ساخن","بارد","ساخنا","باردا"),("المنشفة","ليست","مبللة","جافة","مبللة","جافة"),
          ("الغرفة","ليست","نظيفة","متسخة","نظيفة","متسخة")]

# ---- ANTONYM: cue, antonym (correct), synonym-like foil
ANT_FMT = {"en": 'The opposite of "{c}" is', "fr": 'Le contraire de « {c} » est', "es": 'Lo contrario de "{c}" es',
           "de": 'Das Gegenteil von "{c}" ist', "zh": '“{c}”的反义词是', "ar": 'عكس كلمة "{c}" هو', "hi": '"{c}" का विलोम शब्द है'}
ANT = {
 "en": [("big","small","large"),("fast","slow","quick"),("strong","weak","powerful"),("happy","sad","glad"),("rich","poor","wealthy"),("easy","difficult","simple")],
 "fr": [("grand","petit","énorme"),("rapide","lent","véloce"),("fort","faible","puissant"),("heureux","triste","content"),("riche","pauvre","fortuné"),("facile","difficile","simple")],
 "es": [("grande","pequeño","enorme"),("rápido","lento","veloz"),("fuerte","débil","poderoso"),("feliz","triste","contento"),("rico","pobre","adinerado"),("fácil","difícil","sencillo")],
 "de": [("groß","klein","riesig"),("schnell","langsam","rasch"),("stark","schwach","kräftig"),("glücklich","traurig","froh"),("reich","arm","wohlhabend"),("einfach","schwierig","leicht")],
 "zh": [("大","小","巨大"),("快","慢","迅速"),("强","弱","强大"),("高兴","难过","开心"),("富","穷","富有"),("容易","困难","简单")],
 "ar": [("كبير","صغير","ضخم"),("سريع","بطيء","عاجل"),("قوي","ضعيف","شديد"),("سعيد","حزين","مسرور"),("غني","فقير","ثري"),("سهل","صعب","بسيط")],
 "hi": [("बड़ा","छोटा","विशाल"),("तेज़","धीमा","फुर्तीला"),("मज़बूत","कमज़ोर","ताकतवर"),("खुश","दुखी","प्रसन्न"),("अमीर","गरीब","धनी"),("आसान","मुश्किल","सरल")],
}

# ---- AGREEMENT: head (sg, pl) x attractor PP (sg, pl) x verb (sg, pl). congruent = head and attractor same number
AGR = {
 "en": dict(heads=[("The author","The authors"),("The key","The keys"),("The teacher","The teachers"),("The pilot","The pilots")],
            pps=[("of the book","of the books"),("by the door","by the doors"),("near the car","near the cars")], verbs=[("is","are"),("was","were"),("has","have")]),
 "fr": dict(heads=[("L'auteur","Les auteurs"),("Le professeur","Les professeurs"),("Le pilote","Les pilotes"),("Le directeur","Les directeurs")],
            pps=[("du livre","des livres"),("de la porte","des portes"),("de la voiture","des voitures")], verbs=[("est","sont"),("a","ont"),("était","étaient")]),
 "es": dict(heads=[("El autor","Los autores"),("El profesor","Los profesores"),("El piloto","Los pilotos"),("El director","Los directores")],
            pps=[("del libro","de los libros"),("de la puerta","de las puertas"),("del coche","de los coches")], verbs=[("es","son"),("tiene","tienen"),("estaba","estaban")]),
 "de": dict(heads=[("Der Autor","Die Autoren"),("Der Pilot","Die Piloten"),("Der Direktor","Die Direktoren"),("Der Lehrer","Die Lehrer")],
            pps=[("des Buches","der Bücher"),("der Tür","der Türen"),("des Hauses","der Häuser")], verbs=[("ist","sind"),("hat","haben"),("war","waren")]),
 "ar": dict(heads=[("المدير","المديرون"),("المعلم","المعلمون"),("الطبيب","الأطباء"),("المهندس","المهندسون")],      # human heads only (non-human plurals take feminine singular)
            pps=[("بجانب الباب","بجانب الأبواب"),("أمام المدرسة","أمام المدارس"),("قرب البيت","قرب البيوت")], verbs=[("يعمل","يعملون"),("يعيش","يعيشون"),("يلعب","يلعبون")]),
 "hi": dict(heads=[("लड़का","लड़के"),("कुत्ता","कुत्ते"),("घोड़ा","घोड़े"),("बच्चा","बच्चे")],
            pps=[("मेज़ के पास","मेज़ों के पास"),("दरवाज़े के पास","दरवाज़ों के पास"),("कार के पास","कारों के पास")], verbs=[("है","हैं"),("था","थे")]),
}

# ---- GENDER: prefix has NO gendered determiner (FR notre / ES mi / AR definite article / HI yah). typical=False => opaque noun
# German excluded: predicate adjectives do not inflect, and every agreement route (article, ein/eine, pronoun) leaks gender.
GEN = {
 "fr": dict(fmt="Notre {n} est très", adjs=[("petit","petite"),("grand","grande"),("joli","jolie")],
            nouns=[("livre","m",False),("arbre","m",False),("jardin","m",True),("camion","m",True),("table","f",True),("voiture","f",True),("main","f",False),("nuit","f",False)]),
 "es": dict(fmt="Mi {n} es muy", adjs=[("nuevo","nueva"),("viejo","vieja"),("bonito","bonita")],
            nouns=[("libro","m",True),("coche","m",False),("día","m",False),("mapa","m",False),("casa","f",True),("mesa","f",True),("mano","f",False),("foto","f",False)]),
 "ar": dict(fmt="{n}", adjs=[("جديد","جديدة"),("كبير","كبيرة"),("جميل","جميلة")],
            nouns=[("الباب","m",True),("الكتاب","m",True),("القلم","m",True),("البيت","m",True),("السيارة","f",True),("الطاولة","f",True),("الشمس","f",False),("العين","f",False)]),
 "hi": dict(fmt="यह {n} बहुत", adjs=[("बड़ा","बड़ी"),("अच्छा","अच्छी"),("छोटा","छोटी")],
            nouns=[("दरवाज़ा","m",True),("कमरा","m",True),("घर","m",False),("पानी","m",False),("कुर्सी","f",True),("मेज़","f",False),("किताब","f",False),("रात","f",False)]),
}

# ---- CLASSIFIER (zh only)
CLF = [("书","本"),("桌子","张"),("狗","只"),("车","辆"),("花","朵"),("鱼","条"),("树","棵"),("房子","座")]
CLF_FMT = "昨天我在商店里看到了一些{n}。我买了三"

TASK_LANGS = {"negation": ["en","fr","es","de","zh","ar","hi"], "antonym": ["en","fr","es","de","zh","ar","hi"],
              "agreement": ["en","fr","es","de","ar","hi"], "gender": ["fr","es","ar","hi"], "classifier": ["zh"]}
print("config ok")

config ok


In [2]:
# CELL 2 — tokenizer helpers + item builders (no GPU)
def enc(tok, text): return tok(text, add_special_tokens=False)["input_ids"]
def new_tokens(tok, prefix, cont):
    a, b = enc(tok, prefix), enc(tok, prefix + cont)
    return b[len(a):] if b[:len(a)] == a else None

def mk(task, lang, prefix, correct, foil, **meta): return dict(task=task, lang=lang, prefix=prefix, correct=correct, foil=foil, meta=meta)

def build_negation(lang):
    items = []
    if lang == "ar":
        for s, cop, a, b, aa, ba in NEG_AR:
            items += [mk("negation", lang, f"{s} {cop} {aa}، لذلك {s}", b, a, kind="neg"),
                      mk("negation", lang, f"{s} {cop} {ba}، لذلك {s}", a, b, kind="neg"),
                      mk("negation", lang, f"{s} {a}، لذلك {s}", a, b, kind="aff"),
                      mk("negation", lang, f"{s} {b}، لذلك {s}", b, a, kind="aff")]
        return items
    fneg, faff = NEG_FMT[lang]
    for s1, s2, a, b in NEG[lang]:
        items += [mk("negation", lang, fneg.format(s1=s1, s2=s2, x=a), b, a, kind="neg"),
                  mk("negation", lang, fneg.format(s1=s1, s2=s2, x=b), a, b, kind="neg"),
                  mk("negation", lang, faff.format(s1=s1, s2=s2, x=a), a, b, kind="aff"),
                  mk("negation", lang, faff.format(s1=s1, s2=s2, x=b), b, a, kind="aff")]
    return items

def build_antonym(lang):
    return [mk("antonym", lang, ANT_FMT[lang].format(c=c), ant, syn) for c, ant, syn in ANT[lang]]

def build_agreement(lang):
    d, items, k = AGR[lang], [], 0
    for h_sg, h_pl in d["heads"]:
        for p_sg, p_pl in d["pps"]:
            for hn in ("sg", "pl"):
                for an in ("sg", "pl"):
                    head = h_sg if hn == "sg" else h_pl; pp = p_sg if an == "sg" else p_pl
                    if lang == "hi": prefix = f"{pp} {'का' if hn == 'sg' else 'के'} {head}"
                    else: prefix = f"{head} {pp}"
                    v_sg, v_pl = d["verbs"][k % len(d["verbs"])]; k += 1
                    items.append(mk("agreement", lang, prefix, v_sg if hn == "sg" else v_pl, v_pl if hn == "sg" else v_sg,
                                    congruent=(hn == an), head_num=hn))
    return items

def build_gender(lang):
    d, items = GEN[lang], []
    for n, g, typical in d["nouns"]:
        for m, f in d["adjs"]:
            items.append(mk("gender", lang, d["fmt"].format(n=n), m if g == "m" else f, f if g == "m" else m, typical=typical, gender=g))
    return items

def build_classifier(lang="zh"):
    items, cl = [], [c for _, c in CLF]
    for i, (n, c) in enumerate(CLF):
        for j in (1, 2):
            items.append(mk("classifier", lang, CLF_FMT.format(n=n), c, cl[(i + j) % len(cl)]))
    return items

BUILD = {"negation": build_negation, "antonym": build_antonym, "agreement": build_agreement, "gender": build_gender, "classifier": build_classifier}
ITEMS = {(t, l): BUILD[t](l) for t, ls in TASK_LANGS.items() for l in ls}
print({f"{t}|{l}": len(v) for (t, l), v in ITEMS.items()})

{'negation|en': 20, 'negation|fr': 20, 'negation|es': 20, 'negation|de': 20, 'negation|zh': 20, 'negation|ar': 20, 'negation|hi': 20, 'antonym|en': 6, 'antonym|fr': 6, 'antonym|es': 6, 'antonym|de': 6, 'antonym|zh': 6, 'antonym|ar': 6, 'antonym|hi': 6, 'agreement|en': 48, 'agreement|fr': 48, 'agreement|es': 48, 'agreement|de': 48, 'agreement|ar': 48, 'agreement|hi': 48, 'gender|fr': 24, 'gender|es': 24, 'gender|ar': 24, 'gender|hi': 24, 'classifier|zh': 16}


In [3]:
# CELL 3 — tokenizer-side coverage + candidate collisions (CPU)
import pandas as pd
from transformers import AutoTokenizer
TOK, COVER = {}, []
for key, path in MODELS.items():
    tok = AutoTokenizer.from_pretrained(path); TOK[key] = tok
    for (t, l), items in ITEMS.items():
        sep, ok, coll, difflen = SEP.get(l, " "), 0, 0, 0
        for it in items:
            c, f = new_tokens(tok, it["prefix"], sep + it["correct"]), new_tokens(tok, it["prefix"], sep + it["foil"])
            if c and f:
                ok += 1; coll += int(c[0] == f[0]); difflen += int(len(c) != len(f))
        COVER.append(dict(model=key, task=t, lang=l, n=len(items), scorable=ok / len(items),
                          first_token_collision=coll / max(1, ok), length_differs=difflen / max(1, ok)))
df_cover = pd.DataFrame(COVER)
print(df_cover.round(2).to_string(index=False))
print("\nfirst_token_collision: correct/foil share a first token (first-token scoring would be blind there; full-sequence scoring is used).")
print("length_differs: candidates differ in token count (watch the sum-vs-mean disagreement in the verdict).")

/home/anwoy/miniconda3/envs/ke/lib/python3.10/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


    model       task lang  n  scorable  first_token_collision  length_differs
bloom-1b7   negation   en 20      1.00                   0.00            0.00
bloom-1b7   negation   fr 20      1.00                   0.00            0.20
bloom-1b7   negation   es 20      1.00                   0.00            0.40
bloom-1b7   negation   de 20      1.00                   0.00            0.40
bloom-1b7   negation   zh 20      1.00                   0.00            0.60
bloom-1b7   negation   ar 20      1.00                   0.00            0.80
bloom-1b7   negation   hi 20      1.00                   0.00            0.40
bloom-1b7    antonym   en  6      1.00                   0.00            0.00
bloom-1b7    antonym   fr  6      1.00                   0.00            0.33
bloom-1b7    antonym   es  6      1.00                   0.00            0.33
bloom-1b7    antonym   de  6      1.00                   0.00            0.67
bloom-1b7    antonym   zh  6      0.83                   0.00   

In [4]:
# CELL 4 — score (full-sequence log-prob of each candidate)
import torch, gc
from transformers import AutoModelForCausalLM
dev = "cuda" if torch.cuda.is_available() else "cpu"

@torch.no_grad()
def cand_lp(model, ids_prefix, cand_ids):
    ids = torch.tensor([ids_prefix + cand_ids], device=dev)
    lp = torch.log_softmax(model(ids).logits[0].float(), dim=-1)
    base = len(ids_prefix) - 1
    return [lp[base + j, t].item() for j, t in enumerate(cand_ids)]

RES = []
for key, path in MODELS.items():
    try: model = AutoModelForCausalLM.from_pretrained(path, torch_dtype=torch.bfloat16).to(dev).eval()
    except Exception as e: print("LOAD FAILED", key, e); continue
    tok = TOK[key]; bos = [tok.bos_token_id] if prepend_bos(key) else []
    for (t, l), items in ITEMS.items():
        sep = SEP.get(l, " ")
        for it in items:
            c, f = new_tokens(tok, it["prefix"], sep + it["correct"]), new_tokens(tok, it["prefix"], sep + it["foil"])
            if not (c and f): RES.append(dict(model=key, task=t, lang=l, scored=False, **it["meta"])); continue
            p = bos + enc(tok, it["prefix"])
            lc, lf = cand_lp(model, p, c), cand_lp(model, p, f)
            RES.append(dict(model=key, task=t, lang=l, scored=True, margin_sum=sum(lc) - sum(lf),
                            margin_mean=sum(lc) / len(lc) - sum(lf) / len(lf), **it["meta"]))
    del model; gc.collect(); torch.cuda.empty_cache(); print("done", key)
df_res = pd.DataFrame(RES)

`torch_dtype` is deprecated! Use `dtype` instead!
Loading weights: 100%|██████████| 293/293 [00:00<00:00, 996.19it/s]


done bloom-1b7


Loading weights: 100%|██████████| 365/365 [00:00<00:00, 378.87it/s]


done bloom-7b1


In [5]:
# CELL 5 — verdict
def wilson_lo(k, n, z=1.96):
    if n == 0: return float("nan")
    p = k / n; d = 1 + z * z / n
    return ((p + z * z / (2 * n)) - z * math.sqrt(p * (1 - p) / n + z * z / (4 * n * n))) / d

SPLIT = {"negation": "kind", "agreement": "congruent", "gender": "typical"}
rows = []
for (key, t, l), g in df_res.groupby(["model", "task", "lang"]):
    s = g[g.scored]; n_all, n = len(g), len(s)
    k = int((s.margin_sum > 0).sum()) if n else 0
    r = dict(model=key, task=t, lang=l, n=n_all, scorable=round(n / n_all, 2), acc=round(k / n, 3) if n else float("nan"),
             ci_lo=round(wilson_lo(k, n), 2), acc_mean_norm=round(float((s.margin_mean > 0).mean()), 3) if n else float("nan"),
             mean_margin=round(float(s.margin_sum.mean()), 2) if n else float("nan"))
    if t in SPLIT and n:
        for v, gg in s.groupby(SPLIT[t]): r[f"acc[{SPLIT[t]}={v}]"] = round(float((gg.margin_sum > 0).mean()), 3)
    r["PASS"] = bool(n and n / n_all >= COVER_GATE and k / n >= ACC_GATE and wilson_lo(k, n) > 0.5)
    rows.append(r)
df_verdict = pd.DataFrame(rows).sort_values(["task", "lang", "model"])
pd.set_option("display.width", 250); print(df_verdict.to_string(index=False))
print("\nPASS matrix (all models must pass):")
print(df_verdict.pivot_table(index="task", columns="lang", values="PASS", aggfunc="all").to_string())
df_verdict.to_csv(OUT / "verdict_tasks.csv", index=False); df_cover.to_csv(OUT / "coverage_tasks.csv", index=False); df_res.to_csv(OUT / "item_level_tasks.csv", index=False)

    model       task lang  n  scorable   acc  ci_lo  acc_mean_norm  mean_margin  acc[congruent=False]  acc[congruent=True]  PASS  acc[typical=False]  acc[typical=True]  acc[kind=aff]  acc[kind=neg]
bloom-1b7  agreement   ar 48      1.00 0.604   0.46          0.604         1.62                 0.542                0.667 False                 NaN                NaN            NaN            NaN
bloom-7b1  agreement   ar 48      1.00 0.688   0.55          0.688         1.70                 0.667                0.708 False                 NaN                NaN            NaN            NaN
bloom-1b7  agreement   de 48      1.00 0.583   0.44          0.583         0.87                 0.583                0.583 False                 NaN                NaN            NaN            NaN
bloom-7b1  agreement   de 48      1.00 0.750   0.61          0.750         1.32                 0.708                0.792  True                 NaN                NaN            NaN            NaN
bloom-1b7 